# Hidden-state extraction — DeepSeek-R1-Distill budgets (run `deepseek_r1_qwen7b`)

Stage 3 of the Budget Oracle pipeline, on Colab (needs a GPU: **Runtime → Change runtime type → T4 GPU**).

Budgets were generated by **DeepSeek-R1-Distill-Qwen-7B**. We extract activations from the *small* sibling **DeepSeek-R1-Distill-Qwen-1.5B** (same Qwen2.5-Math-1.5B backbone: 1536 hidden dims, 28 layers) — the oracle's premise is predicting a prover's budget from a cheap model's activations.

Flow: **clone → install → extract → download**. `data/deepseek_r1_qwen7b/problems.jsonl` is committed on the branch, so no upload is needed. Output contract: A3 in `docs/ARTIFACTS.md` (`id`, `type`, `L{n}_mean` arrays + geometry scalars).

In [ ]:
# 1. Clone this branch, install (CPU+GPU deps: torch, transformers, accelerate)
!git clone --branch oracle/DeepSeek-R1-Distill-Qwen-7B https://github.com/newpotatato/smiles-frugalprover.git
%cd smiles-frugalprover
!pip -q install -e ".[gpu]"
!frugalprover info   # sanity: shows torch/transformers versions + CUDA availability

If `data/deepseek_r1_qwen7b/problems.jsonl` is **not** present after cloning (branch not pushed yet), run this cell to upload the file produced locally; otherwise skip it.

In [ ]:
import os
path = "data/deepseek_r1_qwen7b/problems.jsonl"
if not os.path.exists(path):
    os.makedirs("data/deepseek_r1_qwen7b", exist_ok=True)
    from google.colab import files
    up = files.upload()          # choose your local problems.jsonl
    name = next(iter(up))
    if name != path:
        os.replace(name, path)
print("problems.jsonl present:", os.path.exists(path))

In [ ]:
# 2. Extract hidden states with the small DeepSeek model.
#    all_layers + mean pooling (from configs/pipeline.yaml) -> L0..L28 mean vectors + geometry.
#    float16 (T4 has no native bf16); prompt template stays the default "{problem}" (phi(x)).
!frugalprover extract -c configs/pipeline.yaml --run-name deepseek_r1_qwen7b \
    --set extract.model_name=deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --set extract.device=cuda \
    --set extract.dtype=float16 \
    --set extract.batch_size=8 \
    --set extract.max_input_tokens=512

In [ ]:
# 3. Peek at the result, then download the parquet + its sidecar.
import pandas as pd, os
out = "data/deepseek_r1_qwen7b/hidden_states.parquet"
df = pd.read_parquet(out)
print(df.shape, "| MB:", round(os.path.getsize(out)/1e6, 2))
print("cols:", [c for c in df.columns][:6], "...")
print("L0_mean len:", len(df.iloc[0]["L0_mean"]))
from google.colab import files
files.download(out)
files.download(out + ".meta.json")

Put the downloaded `hidden_states.parquet` (+ `.meta.json`) into `data/deepseek_r1_qwen7b/` locally, then run Stage 4–5:

```bash
frugalprover train  -c configs/pipeline.yaml --run-name deepseek_r1_qwen7b --set train.mode=regression
frugalprover report -c configs/pipeline.yaml --run-name deepseek_r1_qwen7b
```